# Notebook 1: Dedicated Prophet Forecasting Pipeline

This notebook implements a standalone Prophet workflow for forecasting monthly credit-card transaction volume for the next 1 to 12 months.

The notebook is designed for two execution modes:

- Run directly in Jupyter for learning, diagnostics, and experimentation.
- Import the reusable functions from `src.prophet_monthly_pipeline` into Streamlit without rewriting notebook logic.

The target is monthly transaction count, not transaction amount or fraud probability.

## Runtime Configuration

`MONTHS_HORIZON` is the only variable a future Streamlit slider needs to control. Keep it between 1 and 12.

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")

DATASET_PATH = "dataset/credit_card_transactions.csv"
TIMESTAMP_COL = "trans_date_trans_time"
MONTHS_HORIZON = 6
VALIDATION_MONTHS = 3

MONTHS_HORIZON = 12
VALIDATION_MONTHS = 3

## 1. Data Ingestion & Preprocessing

The raw CSV is an event table: each row is one transaction. Prophet needs a regular time series, so this step loads the timestamp column, parses dates, counts transactions by calendar month, fills missing months with zero, and removes the incomplete final month.

Removing incomplete months is important. A partial month can look like a sudden demand collapse and distort the model.

In [ ]:
from src.time_series_common import MonthlyDataConfig, load_monthly_transaction_volume

config = MonthlyDataConfig(csv_path=DATASET_PATH, timestamp_col=TIMESTAMP_COL)
monthly_series, profile = load_monthly_transaction_volume(config)

print(f"Rows processed: {profile.row_count:,}")
print(f"Invalid timestamps: {profile.invalid_timestamp_count:,}")
print(f"Training date range: {monthly_series.index.min().date()} to {monthly_series.index.max().date()}")
print(f"Complete monthly observations: {len(monthly_series)}")

display(monthly_series.rename("monthly_transaction_volume").to_frame())

## 2. Feature Engineering / Stationarity Checks

Prophet uses a specific schema: `ds` for dates and `y` for the target. It models trend internally, so we do not manually difference the series or create lag variables.

This cell converts the monthly series into Prophet's expected input format and displays the model-ready frame.

In [ ]:
from src.prophet_monthly_pipeline import to_prophet_frame

prophet_frame = to_prophet_frame(monthly_series)
display(prophet_frame.head())
display(prophet_frame.tail())

## 3. Model Training & Parameter Tuning

This Prophet pipeline runs a compact grid search over `changepoint_prior_scale` and `seasonality_mode`.

- Lower `changepoint_prior_scale` makes trend changes more conservative.
- Higher `changepoint_prior_scale` allows trend to bend more aggressively.
- Additive seasonality is usually safer for count data with stable variance.
- Multiplicative seasonality can help when seasonal swings grow with volume.

The model is evaluated on the most recent validation months, then refit on the full complete monthly history.

In [ ]:
from src.prophet_monthly_pipeline import train_prophet_pipeline

prophet_model = train_prophet_pipeline(
    monthly_series=monthly_series,
    profile=profile,
    validation_months=VALIDATION_MONTHS,
)

print("Best Prophet parameters:")
print(prophet_model["best_params"])

display(prophet_model["metrics"])

## 4. Evaluation Metrics

The metrics table uses the validation period only.

- **MAE** is the average monthly transaction-count miss.
- **RMSE** penalizes unusually large monthly misses more heavily.
- **MAPE** expresses the average miss as a percentage of actual monthly volume.

Lower values are better. A useful forecasting model should beat simple baselines in a broader comparison, but this dedicated notebook focuses on Prophet tuning.

In [ ]:
best_prophet_metrics = prophet_model["metrics"].iloc[0]

print(f"Validation MAE: {best_prophet_metrics['MAE']:,.0f} transactions/month")
print(f"Validation RMSE: {best_prophet_metrics['RMSE']:,.0f} transactions/month")
print(f"Validation MAPE: {best_prophet_metrics['MAPE']:.2f}%")

## 5. Interactive Forecast Inference

The function below is intentionally named `run_forecast(model, months_horizon)` so it can be reused by a Streamlit slider later.

It returns:

- A clean forecast DataFrame.
- A Plotly figure that Streamlit can render with `st.plotly_chart(fig, width="stretch")`.

In [ ]:
from src.prophet_monthly_pipeline import run_forecast

forecast_df, forecast_fig = run_forecast(prophet_model, MONTHS_HORIZON)

display(forecast_df)
forecast_fig.show()

## 6. Model Export

This cell serializes the trained Prophet bundle with pickle. The bundle includes the fitted model, selected hyperparameters, validation metrics, training series, and dataset profile metadata.

In [ ]:
from src.prophet_monthly_pipeline import export_prophet_model

EXPORT_MODEL = True

if EXPORT_MODEL:
    artifact_path = export_prophet_model(prophet_model, "artifacts/prophet_monthly_model.pkl")
    print(f"Model artifact written to: {artifact_path}")

## Developer Insights

### Metric Interpretation

MAE tells you the average number of monthly transactions by which the Prophet forecast missed during validation. If MAE is 5,000, the model was typically off by about 5,000 transactions per month. RMSE is useful when large misses are especially painful because it penalizes those misses more than MAE. MAPE turns the error into a percentage, which is often easier to discuss with business stakeholders.

For this use case, a forecast is accurate enough only if its error is small relative to the monthly operating decision. For example, a 2 percent MAPE may be strong for capacity planning, while a 20 percent MAPE may only be useful for a rough directional view. Always compare Prophet against naive and moving-average baselines before trusting it in production.

### Model Retraining Strategy

Retrain Prophet after each new complete calendar month is available. Append the new raw transaction rows to the dataset, rerun the monthly aggregation, verify that the latest month is complete, rerun hyperparameter tuning, and compare the new validation metrics with the previous run. If the selected parameters change or validation error improves materially, export a new artifact and record the dataset date range used for the model.

In production, schedule this retraining monthly after month-end close. Keep old metrics so you can detect drift: if MAE or MAPE rises for several months, investigate business changes, missing data, seasonality shifts, or a need for a longer validation window.